In [1]:
from tools import search, get_page
from agents import Agent, function_tool

In [2]:
agent_instructions = """

You are a research assistant for Wikipedia topics.

Process:
1) Extract a list of DISTINCT, relevant search terms for the user query using the search() tool. 
2) For EACH term in that list, call the tool get_page() to fetch content for each distinct term. Make at least 3 calls and no more than 7 calls.
3) Do not synthesize an answer until you have called `get_page` on at least 3 distinct terms.
4) After collecting pages, synthesize a concise, accurate answer with references for each section.

Rules:
- You MUST call `get_page` once per term.
- Keep the list short (3–7 terms) and relevant.

You have access to the following tools:
- search - Use this tool to search the Wikipedia API for pertinent topics relating to the full search term provided by the user.
- get_page - Use this tool to search Wikipedia for information regarding pertinent topics provided by the Wikipedia API.

""".strip()



In [3]:
from pydantic import BaseModel, Field
class StructuredWikiAnswer(BaseModel):
    question: str
    answer: str
    references: list[str] = Field(default_factory=list, description="Wikipedia pages included in the answer")
    reasoning_steps: list[str] = Field(default_factory=list, description="Short bullet points outlining the synthesis")


In [ ]:

from jaxn import JSONParserHandler
class StructuredAnswerHandler(JSONParserHandler):
    """Accumulates the parsed JSON value and validates it with StructuredWikiAnswer."""

    def __init__(self):
        self.structured_answer: StructuredWikiAnswer | None = None

    def value(self, value):
        """Called by the parser once a full JSON value has been produced."""
        self.structured_answer = StructuredWikiAnswer.model_validate(value)
        print(f"{value}")



In [5]:
agent_tools = [
    function_tool(search),
    function_tool(get_page)
]

def create_agent():
    agent = Agent(
        name="search assistant",
        tools=agent_tools,
        instructions=agent_instructions,
        output_type=StructuredWikiAnswer,
        model="gpt-4o-mini")
    return agent

search_agent = create_agent()

In [6]:
from agents.exceptions import InputGuardrailTripwireTriggered
from jaxn import StreamingJSONParser
from agents.exceptions import MaxTurnsExceeded
from agents import Runner
from openai.types.responses import ResponseTextDeltaEvent


handler = StructuredAnswerHandler()
parser = StreamingJSONParser(handler)

async def run_stream(agent, input, handler, max_turns=3):
    try:
        result = Runner.run_streamed(
            agent,
            input=input,
            max_turns=max_turns
        )
        
        parser = StreamingJSONParser(handler)

        async for event in result.stream_events():
            if event.type == "run_item_stream_event":
                if event.item.type == "tool_call_item":
                    tool_call = event.item.raw_item
                    f_name = tool_call.name
                    args = tool_call.arguments
                    print(f"TOOL CALL ({event.item.agent.name}): {f_name}({args})")
            
            if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
                parser.parse_incremental(event.data.delta)

        return result
    except MaxTurnsExceeded as e:
        print('too many turns')
        finish_prompt = 'System message: The number of searches has exceeded the limit. Proceed to finishing the writeup'
        finish_message = [{'role': 'user', 'content': finish_prompt}]
        messages = result.to_input_list() + finish_message
        final_result = await run_stream(agent, input=messages, handler=handler, max_turns=1)
        return final_result
    except InputGuardrailTripwireTriggered as e:
        output = e.guardrail_result.output
        if output.tripwire_triggered: 
            print(output.output_info)
        return result

In [7]:
guardrail_instructions = """
Make sure that the question the user asks is about cabybaras. If it's not, report it by
setting `fail` to True.

Explain your decision in the reasoning field, but don't use more than 10 words
""".strip()

class WikiGuardrail(BaseModel):
    reasoning: str
    fail: bool

guardrail_agent = Agent( 
    name="guardrail",
    instructions=guardrail_instructions,
    model='gpt-4o-mini',
    output_type=WikiGuardrail,
)

In [8]:
from agents import input_guardrail, GuardrailFunctionOutput

@input_guardrail
async def documentation_guardrail(ctx, agent, input):
    result = await Runner.run(guardrail_agent, input)
    final_output = result.final_output

    return GuardrailFunctionOutput(
        output_info=final_output.reasoning, 
        tripwire_triggered=final_output.fail,
    )

search_agent = Agent(
    name='search',
    instructions=agent_instructions,
    tools=agent_tools,
    input_guardrails=[documentation_guardrail],
    model="gpt-4o-mini",
    output_type=StructuredWikiAnswer)

In [10]:
from agents.exceptions import InputGuardrailTripwireTriggered
try:
    result = await Runner.run(search_agent, 'whats sqrt(pi)')
except InputGuardrailTripwireTriggered as e:
    output = e.guardrail_result.output
    if output.tripwire_triggered: 
        print(output.output_info)

The question is not about capybaras.
